# Dominik: Causal Tracing na BoolQ (Część I, plan pkt 3)

Zakres (3 pkt), szczegóły w [dominik.md](dominik.md) i [instrukcji](../Zadanie_4.md):

1. Kod pozwalający prześledzić (Causal Tracing) aktywacje dla **8–10 przykładów** z BoolQ: **dowolny przykład na wejściu, dowolne zaszumione tokeny**.
2. Analiza **P("Yes") i P("No")** dla pytania.
3. Zaszumiamy głównie **tokeny przedmiotu pytania** (jednostki nazewnicze).
4. Wykresy i analiza dla **gpt2-xl** i **utahnlp/boolq_gpt2-xl_seed-1**, porównanie modeli i porównanie z pkt 2 (fakty Mikołaja).

**Co możesz importować:** `utils.tracing` (gotowa pętla warstwy × pozycje × strumienie, szum, przywracanie, zapis CSV, `position_summary`, `streams_summary`) i `utils.tracing.plot`. API przyjmuje dowolne `noise_positions`, `restore_positions` i `score_fn` (kontrakt w [README](../README.md#kontrakt-score_fn)).

**Co piszesz sam** (to jest „modyfikacja kodu” z pkt 3a–b), najlepiej w module `boolq.py` obok notebooka, a nie w komórkach:
- wczytanie BoolQ i wybór przykładów,
- format wejścia (gpt2-xl: samo pytanie, `"Question: ...? Answer:"`; utahnlp: passage + pytanie, format wybrany po sprawdzeniu dokładności),
- `score_fn` z P(Yes)/P(No) dla gpt2-xl (logity tokenów `" Yes"`/`" No"`) i dla klasyfikatora utahnlp (softmax po 2 klasach, `1 = True`),
- wybór tokenów do zaszumienia i do przywracania (przy utahnlp przywracaj tylko tokeny pytania + ostatni token, bo inaczej ok. 20 tys. przebiegów na przykład),
- analiza.

Wyniki: `out_dir=artifacts_dir("dominik")` (cache), końcowe PNG do `figures/`. gpt2-xl i klasyfikator ładuj po kolei (razem ok. 13 GB).

In [ ]:
import sys

sys.path.insert(0, "..")

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_formats = ['retina']

In [ ]:
import gc

import pandas as pd
import torch
from datasets import load_dataset

from utils.infra import artifacts_dir, get_device, load_boolq_classifier, load_gpt2_xl, set_seed
from utils.tracing import (
    NOTEBOOK_NOISE_LEVEL_GPT2_XL,
    find_token_positions,
    load_trace_streams,
    position_summary,
    streams_summary,
    tokenize,
    trace_all_streams,
)
from utils.tracing.plot import plot_trace_grid, plot_trace_streams

# from boolq import ...   # TODO: Twój moduł

set_seed(42)
OUT = artifacts_dir("dominik")
FIGURES = "figures"
NOISE = NOTEBOOK_NOISE_LEVEL_GPT2_XL
device = get_device()
device

## 1. Dane BoolQ

In [ ]:
boolq = load_dataset("google/boolq", split="validation")
boolq

TODO: wybór 8–10 przykładów (krótkie pytania z wyraźną jednostką nazewniczą, mieszanka True/False). Pytania są małymi literami i bez `?`.

In [ ]:
# TODO: (nazwa, indeks w boolq, podmiot do zaszumienia)
EXAMPLES = []

## 2. gpt2-xl: format, `score_fn` P(Yes)/P(No), trafność bazowa

In [ ]:
# model, tok = load_gpt2_xl(device)
# TODO: score_fn -> {"p_yes": ..., "p_no": ...}; sprawdź tok.encode(" Yes"), tok.encode(" No")

## 3. gpt2-xl: tracing

In [ ]:
# TODO: trace_all_streams(model, tok, text, noise, score_fn, out_dir=OUT, name=f"{name}__gpt2xl", noise_level=NOISE)

## 4. utahnlp/boolq_gpt2-xl_seed-1: wybór formatu wejścia

In [ ]:
# del model; gc.collect(); torch.mps.empty_cache()
# clf, tok = load_boolq_classifier(device=device)
# TODO: 2–3 formaty na ok. 50 przykładach z walidacji -> dokładność, wybór najlepszego

## 5. utahnlp: tracing

In [ ]:
# TODO: restore_positions = tokeny pytania + ostatni token

## 6. Wykresy

In [ ]:
# TODO: plot_trace_streams(results, key="p_yes", save_path=f"{FIGURES}/<nazwa>.png"); plot_trace_grid(...)

## 7. Porównanie modeli i porównanie z pkt 2 (Mikołaj)

Wyniki Mikołaja: `load_trace_streams(artifacts_dir("mikolaj"), "space_needle")` (i nazwy jego faktów). Jeśli jeszcze ich nie ma, puść sam Space Needle na gpt2-xl tym samym kodem.

In [ ]:
# TODO: streams_summary(...) dla obu modeli i faktów z pkt 2 w jednej tabeli

## Pytania pomocnicze

**Dlaczego wzorce aktywacji dla `boolq_gpt2-xl_seed-1` są nieco inne niż dla gpt2-xl? Na czym polegają różnice?**

TODO

**Czym różnią się wzorce aktywacji od wzorców z pkt 2?**

TODO

## Podsumowanie

TODO